In [2]:
import pandas as pd
import json
import os

In [ ]:
#Intentaré cargar el DataFrame para extraer de aquí los municipios.

df_territories_city=pd.read_csv("./datos_sucios_hito1.csv")
#df_territories_city

#1
#Definiré un nuevo DataFrame, a partir del anterior, para visualizar únicamente los municipios
#city_columns=["Municipio"]
# df_city=pd.read_csv("./datos_sucios_hito1.csv",usecols=city_columns)
# df_city

# 2. Opción ganadora por eficiencia de recursos
# df_city=df_territories_city[["Municipio"]]
# df_city

#3. Sin embargo, puedo definir la lista de municipios directamente
city_list=df_territories_city["Municipio"].unique().tolist()
city_list

#Paralelamente, defino una lista aparte de regiones que tengan acento. En la función, esto me servirá para estandarizar el input del usuario
df_territories_accent=df_territories_city[df_territories_city["Región"].str.contains("á|é|í|ó|ú",case=False)]
df_accent=df_territories_accent["Región"].unique().tolist()

def accent_normalization(text):
    return (text.lower().strip()
            .replace("á","a")
            .replace("é","e")
            .replace("í","i")
            .replace("ó","o")
            .replace("ú","u"))
accent_normalization_dic={accent_normalization(a): a for a in df_accent}

def is_input_valid(text):
    if not text:
        return False
    return text.replace(" ","").isalpha()

#Para mantener la modularidad, no hago instancias dependientes en la función. Si este archivo no existiera, por ejemplo, la función fallaría.
if os.path.exists("progress_territories.json"):
    with open ("progress_territories.json", "r") as f:
        raw_data_saved=json.load(f)
else:
    raw_data_saved={}
data_saved={k.strip():v for k,v in raw_data_saved.items()}

#6. Función principal. Ahora que tengo la lista con los municipios, puedo generar una función para crear el diccionario
def city_territories(city_list,progress=None): #Estoy tratando de crear un diccionario con city como llave y región como valor.

    if progress is None:
        city_territories_dic={}
    else:
        city_territories_dic=progress.copy()
    
    for c in city_list:

        if c in city_territories_dic:
            continue

        while True:
            territories_raw=input(f"Ingrese una región para {c}: \n O ingrese '-' para terminar.").strip()
            
            if territories_raw=="-":
                return city_territories_dic
            
            if is_input_valid(territories_raw):
                break
            print ("La región solo puede contener letras.")


        #territories=accent_normalization.get(territories,territories.capitalize()) #El problema es que no hay separaciòn de datos
        territories_key=accent_normalization(territories_raw)
        territories_processed=accent_normalization_dic.get(territories_key,territories_raw.title())


        print(f"Municipio: {c}, región: {territories_processed}")
        city_territories_dic[c]=territories_processed

    return city_territories_dic

rpoint=city_territories(city_list, progress=data_saved) #Creo el archivo para guardar la información
with open ("progress_territories.json","w") as f:
    json.dump(rpoint,f)

#print(rpoint) esto tiene un problema, porque no me muestra realmente lo que se guardò en el json
print(json.dumps(rpoint, indent=4, ensure_ascii=False))

{
    "Arboletes": "Urabá",
    "Nechí": "Bajo Cauca",
    "Santo Domingo": "Nordeste",
    "Yolombó": "Nordeste",
    "Vegachí": "Nordeste",
    "Cisneros": "Nordeste",
    "Caucasia": "Bajo Cauca",
    "San Juan de Urabá": "Urabá",
    "Turbo": "Urabá",
    "Remedios": "Nordeste",
    "Puerto Claver": "Bajo Cauca",
    "Amalfi": "Nordeste",
    "Piamonte": "Bajo Cauca",
    "Cáceres": "Bajo Cauca",
    "San Pedro de Urabá": "Urabá",
    "Mutatá": "Urabá",
    "Anorí": "Nordeste",
    "Margento": "Bajo Cauca",
    "Cuturú": "Bajo Cauca",
    "Segovia": "Nordeste",
    "Yalí": "Nordeste",
    "El Bagre": "Bajo Cauca",
    "Apartadó": "Urabá",
    "San Roque": "Nordeste",
    "Tarazá": "Bajo Cauca",
    "Chigorodó": "Urabá",
    "Carepa": "Urabá",
    "Murindó": "Urabá",
    "Zaragoza": "Bajo Cauca",
    "Necoclí": "Urabá"
}


In [44]:
#Voy a cargar el archivo original y el json. Pero aquí debo hacer un strip en municipio para que
#coincida con los valores del json. De lo contrario, no habría coincidencia
#Original
df_territories_city["Municipio"]=df_territories_city["Municipio"].str.strip()
df_territories_city

,Región,Municipio,Zona,Contacto,Observaciones,Nombre_Informante,Fecha_Registro
0,Urabá,Arboletes,Urbana,arboletes@dominio.com / 3356676598,Urgente: Riesgo de inundación detectado en el ...,JUAN VALDEZ,18/02/26
1,Nordeste,Nechí,Urbana,NaN,Documentación incompleta.,ALVARO URIBE,2026-02-18
2,Urabá,Santo Domingo,Urbana,Cel: 3981467381 Correo: santo_domingo@dominio.com,Documentación incompleta.,Elena DE LA HOZ,"Feb 18, 2026"
3,Bajo Cauca,Yolombó,Rural,yolombó@dominio.com / 3840553153,Sin novedad en el registro. Todo normal.,pedro picapiedra,18-02-26
4,Occidente,Nechí,Urbana,3969226250,Riesgo detectado en zona norte.,LUISA FERNANDA,2026/02/25
...,...,...,...,...,...,...,...
130,Urabá,Cuturú,Urbana,NaN,Documentación incompleta.,carlos mario,"Feb 18, 2026"
131,Nordeste,Segovia,Rural,NaN,Zona de difícil acceso por lluvias. Importante...,ALVARO URIBE,2026/02/18
132,Nordeste,San Roque,Rural,Cel: 3871122674 Correo: san_roque@dominio.com,Datos recolectados parcialmente.,maria perez,"Feb 18, 2026"
133,Urabá,Carepa,Urbana,3547749439 - carepa@dominio.com,Zona de difícil acceso por lluvias. Importante...,pedro picapiedra,18-02-26


In [45]:
#definitive_region_city=pd.read_json("./progress_territories.json",typ="series") #Si omite typ="series", no cargará el json
#pues técnicamente no es un dataframe, sino un diccionario. Pandas interpreta las llames como columnas y los valores como filas, pero ténicamente eso es un error porque no hay índices.
#de hecho, lo correcto sería usar la librería json

with open("progress_territories.json","r") as f:
    mapping_dict=json.load(f)
print(mapping_dict)

{'Arboletes': 'Urabá', 'Nechí': 'Bajo Cauca', 'Santo Domingo': 'Nordeste', 'Yolombó': 'Nordeste', 'Vegachí': 'Nordeste', 'Cisneros': 'Nordeste', 'Caucasia': 'Bajo Cauca', 'San Juan de Urabá': 'Urabá', 'Turbo': 'Urabá', 'Remedios': 'Nordeste', 'Puerto Claver': 'Bajo Cauca', 'Amalfi': 'Nordeste', 'Piamonte': 'Bajo Cauca', 'Cáceres': 'Bajo Cauca', 'San Pedro de Urabá': 'Urabá', 'Mutatá': 'Urabá', 'Anorí': 'Nordeste', 'Margento': 'Bajo Cauca', 'Cuturú': 'Bajo Cauca', 'Segovia': 'Nordeste', 'Yalí': 'Nordeste', 'El Bagre': 'Bajo Cauca', 'Apartadó': 'Urabá', 'San Roque': 'Nordeste', 'Tarazá': 'Bajo Cauca', 'Chigorodó': 'Urabá', 'Carepa': 'Urabá', 'Murindó': 'Urabá', 'Zaragoza': 'Bajo Cauca', 'Necoclí': 'Urabá'}


In [46]:
#Aquí cruzo el diccionario con la tabla original. Debo recordar que el diccionario tiene como llave
#a las regiones, y que el municipio es valor. Lo mismo debo hacer.
#es decir, en el archivo original, buscar la llave y reemplazar los valores. En este caso
#como no tengo condiciones lógicas, sino una referencia 1 a 1, lo mejor es usar map en vez de np.where
#pues este último sirve mucho para cuando tengo condiciones lógicas

df_territories_city["Región"]=df_territories_city["Municipio"].map(mapping_dict)
df_territories_city

,Región,Municipio,Zona,Contacto,Observaciones,Nombre_Informante,Fecha_Registro
0,Urabá,Arboletes,Urbana,arboletes@dominio.com / 3356676598,Urgente: Riesgo de inundación detectado en el ...,JUAN VALDEZ,18/02/26
1,Bajo Cauca,Nechí,Urbana,NaN,Documentación incompleta.,ALVARO URIBE,2026-02-18
2,Nordeste,Santo Domingo,Urbana,Cel: 3981467381 Correo: santo_domingo@dominio.com,Documentación incompleta.,Elena DE LA HOZ,"Feb 18, 2026"
3,Nordeste,Yolombó,Rural,yolombó@dominio.com / 3840553153,Sin novedad en el registro. Todo normal.,pedro picapiedra,18-02-26
4,Bajo Cauca,Nechí,Urbana,3969226250,Riesgo detectado en zona norte.,LUISA FERNANDA,2026/02/25
...,...,...,...,...,...,...,...
130,Bajo Cauca,Cuturú,Urbana,NaN,Documentación incompleta.,carlos mario,"Feb 18, 2026"
131,Nordeste,Segovia,Rural,NaN,Zona de difícil acceso por lluvias. Importante...,ALVARO URIBE,2026/02/18
132,Nordeste,San Roque,Rural,Cel: 3871122674 Correo: san_roque@dominio.com,Datos recolectados parcialmente.,maria perez,"Feb 18, 2026"
133,Urabá,Carepa,Urbana,3547749439 - carepa@dominio.com,Zona de difícil acceso por lluvias. Importante...,pedro picapiedra,18-02-26


In [47]:
#Guardo la versión corregida: si quisiera sobreescribirla, simplemente pongo el mismo nombre
df_territories_city.to_csv("/home/jose-manuel-betancur/Escritorio/Hito1/datos_limpios.csv",index=False,encoding="utf-8")

Próxima tarea: que los datos se reemplacen efectivamente en el archivo. Crear una copia del archivo original para contrastar.